# structured-v16 smoke-7

Launch this on Kaggle. Do not submit it to the leaderboard.

It runs the plain packed zip of `agents/structured-v16` on seven tasks.
Nothing is registered. The hosted runner calls `compile_submission` with the
host tool registry, model registry, and code executor, and it does not pass a
callback registry. This notebook does not wrap that call. Unpack the zip and
point the official starter at that directory. Traces stay on disk.

The report cell prints resolved count, per-task wall time, no-edit runs,
calls to the first edit, whether the text `HYPOTHESIS:` appeared, whether
repair_1 returned before the 300 second cap, and the elapsed time of that
hand-off. `rich_3105` and `rich_3938` are marked effectively unreachable from
the never-edit write-up. That mark is a judgment, not a filter, and it is not a
score. This notebook was not executed in the lab checkout.


In [ ]:
import os
from pathlib import Path

TASK_IDS = [
    "rich_3934",
    "rich_3105",
    "rich_3938",
    "rich_3472",
    "rich_3944",
    "fastapi_11194",
    "rich_3454",
]
EFFECTIVELY_UNREACHABLE = {"rich_3105", "rich_3938"}
AGENT_DIR = Path(os.environ.get("AGENT_DIR", "/kaggle/working/agent"))
print("structured-v16", AGENT_DIR)
print("tasks", len(TASK_IDS))
print("plain zip; no callback registry")

In [ ]:
def compile_plain(submission_dir, tool_registry, model_registry, code_executor):
    """Same arguments the hosted runner passes. No callback registry."""
    from adk_submission import compile_submission

    return compile_submission(
        submission_dir=submission_dir,
        tool_registry=tool_registry,
        model_registry=model_registry,
        code_executor=code_executor,
    )


print("compile_plain is the hosted call. The starter runs it on the unpacked zip.")

## After the official loop

Point `RESULTS` at the run directory that holds `task_results.jsonl` and
`results/traces/`. This cell does not start the model. Traces stay on disk.


In [ ]:
import json
import os
from pathlib import Path

EFFECTIVELY_UNREACHABLE = {"rich_3105", "rich_3938"}
EDIT_MARKERS = ("line_fix", "cat >", "cat>>", "tee ")
LATER_ROLES = {"verify_1", "repair_2", "verify_2", "repair_3", "verify_3"}
CAP_SECONDS = 300


def _events(trace):
    if isinstance(trace, dict):
        if isinstance(trace.get("steps"), list):
            return trace["steps"]
        return trace.get("events") or trace.get("trace") or []
    return []


def _author(event):
    if not isinstance(event, dict):
        return ""
    extra = event.get("extra") if isinstance(event.get("extra"), dict) else {}
    for source in (event, extra):
        for key in ("author", "agent", "agent_name", "role"):
            value = source.get(key)
            if value:
                return str(value)
    return ""


def _elapsed(event):
    if not isinstance(event, dict):
        return None
    extra = event.get("extra") if isinstance(event.get("extra"), dict) else {}
    for source in (event, extra):
        for key in ("elapsed_s", "agent_elapsed_seconds", "elapsed_seconds", "elapsed"):
            value = source.get(key)
            if value is None:
                continue
            try:
                return float(value)
            except (TypeError, ValueError):
                continue
    return None


def _tool_name(event):
    if not isinstance(event, dict):
        return ""
    return str(event.get("tool") or event.get("name") or event.get("tool_name") or "")


def _blob(event):
    return json.dumps(event, default=str)


def _is_edit(event):
    name = _tool_name(event)
    if name in {"edit_file", "write_file"}:
        return True
    return any(marker in _blob(event) for marker in EDIT_MARKERS)


def _hypothesis(event):
    return "HYPOTHESIS:" in _blob(event)


def repair_1_handoff(trace, wall):
    """Whether repair_1 returned before the cap, and the hand-off elapsed."""
    handoff = None
    returned = False
    last_repair = None
    for event in _events(trace):
        author = _author(event)
        elapsed = _elapsed(event)
        if author == "repair_1" and elapsed is not None:
            last_repair = elapsed
        if author in LATER_ROLES:
            returned = True
            handoff = last_repair if last_repair is not None else elapsed
            break
    if not returned:
        return {"returned_before_cap": False, "handoff_elapsed": None}
    before_cap = handoff is None or handoff < CAP_SECONDS
    if wall is not None and handoff is None:
        try:
            before_cap = float(wall) < CAP_SECONDS
        except (TypeError, ValueError):
            before_cap = True
    return {"returned_before_cap": before_cap, "handoff_elapsed": handoff}


def summarize(rows, traces):
    tasks = []
    for row in rows:
        task_id = row.get("instance_id") or row.get("task_id") or ""
        trace = traces.get(task_id) or {}
        events = _events(trace)
        calls = 0
        first = None
        edited = False
        hypothesis = False
        for event in events:
            if _tool_name(event):
                calls += 1
            if _hypothesis(event):
                hypothesis = True
            if first is None and _is_edit(event):
                first = calls
                edited = True
        if not edited:
            patch = str(row.get("patch") or row.get("model_patch") or "")
            edited = bool(patch.strip())
        wall = row.get("wall_seconds") or row.get("duration_seconds")
        handoff = repair_1_handoff(trace, wall)
        tasks.append(
            {
                "instance_id": task_id,
                "resolved": bool(row.get("resolved") or row.get("passed")),
                "wall_seconds": wall,
                "no_edit": not edited,
                "calls_to_first_edit": first,
                "hypothesis": hypothesis,
                "repair_1_returned_before_cap": handoff["returned_before_cap"],
                "handoff_elapsed": handoff["handoff_elapsed"],
                "effectively_unreachable": task_id in EFFECTIVELY_UNREACHABLE,
            }
        )
    return {
        "n": len(tasks),
        "resolved": sum(item["resolved"] for item in tasks),
        "no_edit_runs": sum(item["no_edit"] for item in tasks),
        "hypothesis_runs": sum(item["hypothesis"] for item in tasks),
        "repair_1_returned_before_cap": sum(item["repair_1_returned_before_cap"] for item in tasks),
        "tasks": tasks,
    }


RESULTS = Path(os.environ.get("V16_RESULTS", "/kaggle/working"))
rows = []
results_path = RESULTS / "task_results.jsonl"
if results_path.is_file():
    for line in results_path.read_text().splitlines():
        if line.strip():
            rows.append(json.loads(line))
traces = {}
trace_dir = RESULTS / "results" / "traces"
if not trace_dir.is_dir():
    trace_dir = RESULTS / "traces"
if trace_dir.is_dir():
    for path in sorted(trace_dir.glob("*.json")):
        payload = json.loads(path.read_text())
        task_id = payload.get("instance_id") or path.stem.removeprefix("trace_")
        traces[task_id] = payload
report = summarize(rows, traces)
print("resolved", report["resolved"], "of", report["n"])
print("no_edit_runs", report["no_edit_runs"])
print("hypothesis_runs", report["hypothesis_runs"])
print("repair_1_returned_before_cap", report["repair_1_returned_before_cap"])
for item in report["tasks"]:
    print(
        item["instance_id"],
        "resolved" if item["resolved"] else "unresolved",
        "wall",
        item["wall_seconds"],
        "no_edit" if item["no_edit"] else "edit",
        "calls_to_first_edit",
        item["calls_to_first_edit"],
        "HYPOTHESIS" if item["hypothesis"] else "no_HYPOTHESIS",
        "repair_1_returned_before_cap"
        if item["repair_1_returned_before_cap"]
        else "repair_1_held_cap",
        "handoff_elapsed",
        item["handoff_elapsed"],
        "unreachable_judgment" if item["effectively_unreachable"] else "reachable_judgment",
    )